In [1]:
import pandas as pd

# Replace 'path_to_your_file.csv' with the actual file path of your CSV file
file_path = '/Users/lkimball/Desktop/Wesleyan_Tennis_Analytics/singles_matches_df.csv'
data = pd.read_csv(file_path)

# To view the first few rows of the dataset and check its structure
data.head()


,Name,Position,Duration,1 - Event,2 - Player Name,3 - Opponent Name,4 - Surface,A1: Server,A2: Serve Data,A3: Serve Placement,...,x: Player X,x: Player Y,zz - Rally Data,zz - Returner,J4: Poach Returning Team,F5: Point Penalty,H3: Court Position,x: Player A Set Score,x: Player B Set Score,z - 1st Serve Fault Target
0,0-0 (1),0,13033,2023 Wesleyan Invitational,Brett Keeling,Lefteri Apostolou,Hard,Brett Keeling,1st Serve Made,Wide,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0-15 (1),13033,15433,2023 Wesleyan Invitational,Brett Keeling,Lefteri Apostolou,Hard,Brett Keeling,2nd Serve Made,Wide,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,15-15 (1),28467,11200,2023 Wesleyan Invitational,Brett Keeling,Lefteri Apostolou,Hard,Brett Keeling,1st Serve Made,Body,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,15-30 (1),39667,17800,2023 Wesleyan Invitational,Brett Keeling,Lefteri Apostolou,Hard,Brett Keeling,1st Serve Made,Wide,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,30-30 (1),57467,7700,2023 Wesleyan Invitational,Brett Keeling,Lefteri Apostolou,Hard,Brett Keeling,1st Serve Made,Body,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# Filter the data based on the player names provided
players = [
    "Ben Mitchell", "Leila Epstein", "Lane Durkin", "Jan Hrazdil", "Sarah Youngberg", "Chase Kasday", "Alexander Bernard",
    "Patrick Kypson", "Carlos Alcaraz Garfia", "Jackie Soloveychik", "JT Bilski", "Max Lustgarten", "Harrison Collomb",
    "Nicolas Kotzen", "Brett Keeling", "Michael Zheng", "Alexandra Burnett", "Caitlyn Ferrante", 'Eudice Chong'
]

# Calculate the % of return points won for each player
return_points_data = []

for player in players:
    total_returns = data[data['B1: Returner'] == player]
    points_won = total_returns[total_returns['F1: Point Won'] == player]
    if len(total_returns) > 0:
        percentage_won = len(points_won) / len(total_returns) * 100
    else:
        percentage_won = 0  # To handle division by zero in case there are no return records for a player
    return_points_data.append({'Player': player, 'Return Points Won (%)': percentage_won})

return_points_df = pd.DataFrame(return_points_data)

In [3]:
# Initialize a list to hold the data
return_points_data = []

# Calculate the total and percentage of return points won for each player
for player in players:
    total_returns = data[data['B1: Returner'] == player]
    points_won = total_returns[total_returns['F1: Point Won'] == player]
    total_points = len(total_returns)
    percentage_won = (len(points_won) / total_points * 100) if total_points > 0 else 0

    return_points_data.append({
        'Player': player,
        'Total Return Points': total_points,
        'Return Points Won (%)': percentage_won
    })

# Convert the list to a DataFrame
return_points_df = pd.DataFrame(return_points_data)

# Display the DataFrame
return_points_df.head()

,Player,Total Return Points,Return Points Won (%)
0,Ben Mitchell,313,46.006390
1,Leila Epstein,374,51.069519
2,Lane Durkin,195,55.897436
3,Jan Hrazdil,102,50.000000
4,Sarah Youngberg,194,59.278351


In [4]:
# Update the calculation for return points won against 1st serves to consider only points where the serve was a 1st serve
return_points_data_corrected = []

for player in players:
    player_returns = data[data['B1: Returner'] == player]
    points_won = player_returns[player_returns['F1: Point Won'] == player]
    total_points = len(player_returns)
    
    # Filter for 1st serve points
    first_serve_returns = player_returns[player_returns['A2: Serve Data'].isin(['1st Serve Made', '1st Serve Ace'])]
    points_won_against_1st_serve = first_serve_returns[first_serve_returns['F1: Point Won'] == player]
    total_first_serve_points = len(first_serve_returns)
    
    percentage_won = (len(points_won) / total_points * 100) if total_points > 0 else 0
    percentage_won_1st_serve = (len(points_won_against_1st_serve) / total_first_serve_points * 100) if total_first_serve_points > 0 else 0
    
    return_points_data_corrected.append({
        'Player': player,
        'Total Return Points': total_points,
        'Return Points Won (%)': percentage_won,
        'Return Points Won Against 1st Serve (%)': percentage_won_1st_serve,
        'Total 1st Serve Return Points': total_first_serve_points
    })

# Create a new DataFrame to include the corrected return points won against 1st serves
return_points_df_corrected = pd.DataFrame(return_points_data_corrected)
return_points_df_corrected.head()


,Player,Total Return Points,Return Points Won (%),Return Points Won Against 1st Serve (%),Total 1st Serve Return Points
0,Ben Mitchell,313,46.006390,35.820896,201
1,Leila Epstein,374,51.069519,46.120690,232
2,Lane Durkin,195,55.897436,48.000000,125
3,Jan Hrazdil,102,50.000000,43.859649,57
4,Sarah Youngberg,194,59.278351,55.660377,106


In [5]:
final_data = []

for player in players:
    player_returns = data[data['B1: Returner'] == player]
    first_serve_returns = player_returns[player_returns['A2: Serve Data'].isin(['1st Serve Made', '1st Serve Ace'])]

    # Points won on 1st serve
    points_won_against_1st_serve = first_serve_returns[first_serve_returns['F1: Point Won'] == player]
    
    # Ad court 1st serve returns
    ad_court_returns = first_serve_returns[first_serve_returns['H2: Deuce Ad'] == 'Ad Court']
    ad_court_points_won = ad_court_returns[ad_court_returns['F1: Point Won'] == player]

    # Deuce court 1st serve returns
    deuce_court_returns = first_serve_returns[first_serve_returns['H2: Deuce Ad'] == 'Deuce Court']
    deuce_court_points_won = deuce_court_returns[deuce_court_returns['F1: Point Won'] == player]

    # Calculate percentages
    total_first_serve_points = len(first_serve_returns)
    percentage_won_against_1st_serve = (len(points_won_against_1st_serve) / total_first_serve_points * 100) if total_first_serve_points > 0 else 0
    ad_court_percentage_won = (len(ad_court_points_won) / len(ad_court_returns) * 100) if len(ad_court_returns) > 0 else 0
    deuce_court_percentage_won = (len(deuce_court_points_won) / len(deuce_court_returns) * 100) if len(deuce_court_returns) > 0 else 0

    final_data.append({
        'Player': player,
        'Total Return Points': len(player_returns),
        'Return Points Won (%)': len(points_won_against_1st_serve) / len(player_returns) * 100,
        'Total 1st Serve Return Points': total_first_serve_points,
        'Return Points Won Against 1st Serve (%)': percentage_won_against_1st_serve,
        'Ad Court 1st Serve Return Win (%)': ad_court_percentage_won,
        'Deuce Court 1st Serve Return Win (%)': deuce_court_percentage_won
    })

final_return_points_df = pd.DataFrame(final_data)
final_return_points_df.head()


,Player,Total Return Points,Return Points Won (%),Total 1st Serve Return Points,Return Points Won Against 1st Serve (%),Ad Court 1st Serve Return Win (%),Deuce Court 1st Serve Return Win (%)
0,Ben Mitchell,313,23.003195,201,35.820896,34.259259,37.634409
1,Leila Epstein,374,28.609626,232,46.120690,48.672566,43.697479
2,Lane Durkin,195,30.769231,125,48.000000,44.444444,51.612903
3,Jan Hrazdil,102,24.509804,57,43.859649,44.000000,43.750000
4,Sarah Youngberg,194,30.412371,106,55.660377,60.000000,50.000000


In [6]:
# Final adjustments to ensure all calculations and column values are accurate
final_adjusted_data = []

for player in players:
    player_returns = data[data['B1: Returner'] == player]
    first_serve_returns = player_returns[player_returns['A2: Serve Data'].isin(['1st Serve Made', '1st Serve Ace'])]

    # Filter for Ad and Deuce court 1st serve returns
    ad_court_returns = first_serve_returns[first_serve_returns['H2: Deuce Ad'] == 'Ad Court']
    deuce_court_returns = first_serve_returns[first_serve_returns['H2: Deuce Ad'] == 'Deuce Court']

    # Points won on 1st serve
    points_won = player_returns[player_returns['F1: Point Won'] == player]
    points_won_against_1st_serve = first_serve_returns[first_serve_returns['F1: Point Won'] == player]
    ad_court_points_won = ad_court_returns[ad_court_returns['F1: Point Won'] == player]
    deuce_court_points_won = deuce_court_returns[deuce_court_returns['F1: Point Won'] == player]

    # Calculate percentages
    total_points = len(player_returns)
    total_first_serve_points = len(first_serve_returns)
    total_ad_court_points = len(ad_court_returns)
    total_deuce_court_points = len(deuce_court_returns)
    percentage_won = (len(points_won) / total_points * 100) if total_points > 0 else 0
    percentage_won_against_1st_serve = (len(points_won_against_1st_serve) / total_first_serve_points * 100) if total_first_serve_points > 0 else 0
    ad_court_percentage_won = (len(ad_court_points_won) / total_ad_court_points * 100) if total_ad_court_points > 0 else 0
    deuce_court_percentage_won = (len(deuce_court_points_won) / total_deuce_court_points * 100) if total_deuce_court_points > 0 else 0

    final_adjusted_data.append({
        'Player': player,
        'Total Return Points': total_points,
        'Return Points Won (%)': percentage_won,
        'Total 1st Serve Return Points': total_first_serve_points,
        'Return Points Won Against 1st Serve (%)': percentage_won_against_1st_serve,
        'Ad Court Total 1st Serve Returns': total_ad_court_points,
        'Ad Court 1st Serve Return Win (%)': ad_court_percentage_won,
        'Deuce Court Total 1st Serve Returns': total_deuce_court_points,
        'Deuce Court 1st Serve Return Win (%)': deuce_court_percentage_won
    })

# Create the final DataFrame with the adjusted data
final_adjusted_return_points_df = pd.DataFrame(final_adjusted_data)
final_adjusted_return_points_df


,Player,Total Return Points,Return Points Won (%),Total 1st Serve Return Points,Return Points Won Against 1st Serve (%),Ad Court Total 1st Serve Returns,Ad Court 1st Serve Return Win (%),Deuce Court Total 1st Serve Returns,Deuce Court 1st Serve Return Win (%)
0,Ben Mitchell,313,46.006390,201,35.820896,108,34.259259,93,37.634409
1,Leila Epstein,374,51.069519,232,46.120690,113,48.672566,119,43.697479
2,Lane Durkin,195,55.897436,125,48.000000,63,44.444444,62,51.612903
3,Jan Hrazdil,102,50.000000,57,43.859649,25,44.000000,32,43.750000
4,Sarah Youngberg,194,59.278351,106,55.660377,60,60.000000,46,50.000000
5,Chase Kasday,123,41.463415,97,44.329897,50,54.000000,47,34.042553
6,Alexander Bernard,169,50.887574,106,39.622642,46,32.608696,60,45.000000
7,Patrick Kypson,70,38.571429,37,18.918919,21,23.809524,16,12.500000
8,Carlos Alcaraz Garfia,128,34.375000,73,32.876712,38,36.842105,35,28.571429
9,Jackie Soloveychik,51,64.705882,36,58.333333,17,58.823529,19,57.894737


In [7]:
# Extending the analysis to include only records where the returner's name matches the player name
# for Forehand first serve returns and calculating the percentage won
fh_return_data = []

for player in players:
    player_returns = data[data['B1: Returner'] == player]
    fh_first_serve_returns = player_returns[(player_returns['A2: Serve Data'].isin(['1st Serve Made', '1st Serve Ace'])) &
                                            (player_returns['B2: Return Data'].str.contains('Forehand'))]

    # Points won on FH first serve
    fh_points_won = fh_first_serve_returns[fh_first_serve_returns['F1: Point Won'] == player]

    # Calculate total and percentage
    total_fh_first_serve_returns = len(fh_first_serve_returns)
    fh_percentage_won = (len(fh_points_won) / total_fh_first_serve_returns * 100) if total_fh_first_serve_returns > 0 else 0

    fh_return_data.append({
        'Player': player,
        'Total FH 1st Serve Returns': total_fh_first_serve_returns,
        'FH 1st Serve Return Win (%)': fh_percentage_won
    })

# Create a new DataFrame with the FH 1st serve return data
fh_return_points_df = pd.DataFrame(fh_return_data)
fh_return_points_df.head()


,Player,Total FH 1st Serve Returns,FH 1st Serve Return Win (%)
0,Ben Mitchell,75,32.000000
1,Leila Epstein,142,49.295775
2,Lane Durkin,66,46.969697
3,Jan Hrazdil,34,44.117647
4,Sarah Youngberg,40,50.000000


In [8]:
# Merge the forehand return data with the final adjusted return points DataFrame
merged_data = pd.merge(final_adjusted_return_points_df, fh_return_points_df, on='Player', how='left')

# Renaming columns for clarity
merged_data.rename(columns={
    'Total FH 1st Serve Returns': 'FH 1st Serve Returns Total',
    'FH 1st Serve Return Win (%)': 'FH 1st Serve Returns Win %'
}, inplace=True)

merged_data.head()


,Player,Total Return Points,Return Points Won (%),Total 1st Serve Return Points,Return Points Won Against 1st Serve (%),Ad Court Total 1st Serve Returns,Ad Court 1st Serve Return Win (%),Deuce Court Total 1st Serve Returns,Deuce Court 1st Serve Return Win (%),FH 1st Serve Returns Total,FH 1st Serve Returns Win %
0,Ben Mitchell,313,46.006390,201,35.820896,108,34.259259,93,37.634409,75,32.000000
1,Leila Epstein,374,51.069519,232,46.120690,113,48.672566,119,43.697479,142,49.295775
2,Lane Durkin,195,55.897436,125,48.000000,63,44.444444,62,51.612903,66,46.969697
3,Jan Hrazdil,102,50.000000,57,43.859649,25,44.000000,32,43.750000,34,44.117647
4,Sarah Youngberg,194,59.278351,106,55.660377,60,60.000000,46,50.000000,40,50.000000


In [9]:
# Calculate the percentage of FH 1st serve returns in play (excluding errors)
fh_in_play_data = []

for player in players:
    player_returns = data[data['B1: Returner'] == player]
    fh_first_serve_returns = player_returns[(player_returns['A2: Serve Data'].isin(['1st Serve Made', '1st Serve Ace'])) &
                                            (player_returns['B2: Return Data'].str.contains('Forehand'))]
    fh_errors = fh_first_serve_returns[fh_first_serve_returns['B2: Return Data'] == 'Forehand Return Error']

    # Calculate total and percentage of FH returns in play (not errors)
    total_fh_first_serve_returns = len(fh_first_serve_returns)
    total_fh_errors = len(fh_errors)
    fh_in_play = total_fh_first_serve_returns - total_fh_errors
    fh_in_play_percentage = (fh_in_play / total_fh_first_serve_returns * 100) if total_fh_first_serve_returns > 0 else 0

    fh_in_play_data.append({
        'Player': player,
        'FH 1st Serve Returns Total': total_fh_first_serve_returns,
        'FH 1st Serve Returns In Play (%)': fh_in_play_percentage
    })

# Create a new DataFrame with the FH 1st serve returns in play data
fh_in_play_df = pd.DataFrame(fh_in_play_data)
fh_in_play_df.head()


,Player,FH 1st Serve Returns Total,FH 1st Serve Returns In Play (%)
0,Ben Mitchell,75,80.000000
1,Leila Epstein,142,85.211268
2,Lane Durkin,66,75.757576
3,Jan Hrazdil,34,64.705882
4,Sarah Youngberg,40,87.500000


In [10]:
# Merge the FH in play data with the merged data DataFrame
final_data_complete = pd.merge(merged_data, fh_in_play_df[['Player', 'FH 1st Serve Returns In Play (%)']], on='Player', how='left')

final_data_complete.head()


,Player,Total Return Points,Return Points Won (%),Total 1st Serve Return Points,Return Points Won Against 1st Serve (%),Ad Court Total 1st Serve Returns,Ad Court 1st Serve Return Win (%),Deuce Court Total 1st Serve Returns,Deuce Court 1st Serve Return Win (%),FH 1st Serve Returns Total,FH 1st Serve Returns Win %,FH 1st Serve Returns In Play (%)
0,Ben Mitchell,313,46.006390,201,35.820896,108,34.259259,93,37.634409,75,32.000000,80.000000
1,Leila Epstein,374,51.069519,232,46.120690,113,48.672566,119,43.697479,142,49.295775,85.211268
2,Lane Durkin,195,55.897436,125,48.000000,63,44.444444,62,51.612903,66,46.969697,75.757576
3,Jan Hrazdil,102,50.000000,57,43.859649,25,44.000000,32,43.750000,34,44.117647,64.705882
4,Sarah Youngberg,194,59.278351,106,55.660377,60,60.000000,46,50.000000,40,50.000000,87.500000


In [11]:
# Adjusting the code to handle NA values when filtering Forehand returns for the placement analysis
fh_return_placement_data = []

for player in players:
    player_returns = data[data['B1: Returner'] == player]
    fh_returns = player_returns[player_returns['B2: Return Data'].fillna('').str.contains('Forehand')]

    # Total FH returns for the player
    total_fh_returns = len(fh_returns)

    # Count each type of return placement
    fh_long = fh_returns['B3: Return Placement'].str.contains('Long', na=False).sum()
    fh_net = fh_returns['B3: Return Placement'].str.contains('Net', na=False).sum()
    fh_wide = fh_returns['B3: Return Placement'].str.contains('Wide', na=False).sum()
    fh_deep = fh_returns['B3: Return Placement'].str.contains('Deep', na=False).sum()

    # Calculate percentages
    fh_long_percentage = (fh_long / total_fh_returns * 100) if total_fh_returns > 0 else 0
    fh_net_percentage = (fh_net / total_fh_returns * 100) if total_fh_returns > 0 else 0
    fh_wide_percentage = (fh_wide / total_fh_returns * 100) if total_fh_returns > 0 else 0
    fh_deep_percentage = (fh_deep / total_fh_returns * 100) if total_fh_returns > 0 else 0

    fh_return_placement_data.append({
        'Player': player,
        'FH Returns Long (%)': fh_long_percentage,
        'FH Returns Net (%)': fh_net_percentage,
        'FH Returns Wide (%)': fh_wide_percentage,
        'FH Returns Deep (%)': fh_deep_percentage
    })

# Create a new DataFrame with the FH return placement data
fh_return_placement_df = pd.DataFrame(fh_return_placement_data)
fh_return_placement_df.head()


,Player,FH Returns Long (%),FH Returns Net (%),FH Returns Wide (%),FH Returns Deep (%)
0,Ben Mitchell,6.250000,6.250000,2.678571,33.928571
1,Leila Epstein,5.668016,6.882591,2.834008,42.510121
2,Lane Durkin,5.607477,10.280374,3.738318,36.448598
3,Jan Hrazdil,10.344828,10.344828,6.896552,32.758621
4,Sarah Youngberg,3.409091,6.818182,2.272727,43.181818


In [12]:
# Merge the FH return placement data with the final comprehensive data DataFrame
final_data_with_placement = pd.merge(final_data_complete, fh_return_placement_df, on='Player', how='left')

final_data_with_placement.head()


,Player,Total Return Points,Return Points Won (%),Total 1st Serve Return Points,Return Points Won Against 1st Serve (%),Ad Court Total 1st Serve Returns,Ad Court 1st Serve Return Win (%),Deuce Court Total 1st Serve Returns,Deuce Court 1st Serve Return Win (%),FH 1st Serve Returns Total,FH 1st Serve Returns Win %,FH 1st Serve Returns In Play (%),FH Returns Long (%),FH Returns Net (%),FH Returns Wide (%),FH Returns Deep (%)
0,Ben Mitchell,313,46.006390,201,35.820896,108,34.259259,93,37.634409,75,32.000000,80.000000,6.250000,6.250000,2.678571,33.928571
1,Leila Epstein,374,51.069519,232,46.120690,113,48.672566,119,43.697479,142,49.295775,85.211268,5.668016,6.882591,2.834008,42.510121
2,Lane Durkin,195,55.897436,125,48.000000,63,44.444444,62,51.612903,66,46.969697,75.757576,5.607477,10.280374,3.738318,36.448598
3,Jan Hrazdil,102,50.000000,57,43.859649,25,44.000000,32,43.750000,34,44.117647,64.705882,10.344828,10.344828,6.896552,32.758621
4,Sarah Youngberg,194,59.278351,106,55.660377,60,60.000000,46,50.000000,40,50.000000,87.500000,3.409091,6.818182,2.272727,43.181818


In [13]:
# Calculate statistics for Backhand (BH) returns on 1st serves
bh_return_data = []

for player in players:
    player_returns = data[data['B1: Returner'] == player]
    # Filter for BH returns on 1st serves
    bh_first_serve_returns = player_returns[(player_returns['A2: Serve Data'].isin(['1st Serve Made', '1st Serve Ace'])) &
                                            (player_returns['B2: Return Data'].fillna('').str.contains('Backhand'))]

    # Total BH 1st serve returns for the player
    total_bh_first_serve_returns = len(bh_first_serve_returns)

    # Points won on BH 1st serve
    bh_points_won = bh_first_serve_returns[bh_first_serve_returns['F1: Point Won'] == player]
    bh_win_percentage = (len(bh_points_won) / total_bh_first_serve_returns * 100) if total_bh_first_serve_returns > 0 else 0

    # BH returns in play (excluding errors)
    bh_in_play = bh_first_serve_returns[~bh_first_serve_returns['B2: Return Data'].str.contains('Error', na=False)]
    bh_in_play_percentage = (len(bh_in_play) / total_bh_first_serve_returns * 100) if total_bh_first_serve_returns > 0 else 0

    # BH return placements
    bh_long = bh_first_serve_returns['B3: Return Placement'].str.contains('Long', na=False).sum()
    bh_net = bh_first_serve_returns['B3: Return Placement'].str.contains('Net', na=False).sum()
    bh_wide = bh_first_serve_returns['B3: Return Placement'].str.contains('Wide', na=False).sum()
    bh_deep = bh_first_serve_returns['B3: Return Placement'].str.contains('Deep', na=False).sum()

    # Calculate percentages for return placements
    bh_long_percentage = (bh_long / total_bh_first_serve_returns * 100) if total_bh_first_serve_returns > 0 else 0
    bh_net_percentage = (bh_net / total_bh_first_serve_returns * 100) if total_bh_first_serve_returns > 0 else 0
    bh_wide_percentage = (bh_wide / total_bh_first_serve_returns * 100) if total_bh_first_serve_returns > 0 else 0
    bh_deep_percentage = (bh_deep / total_bh_first_serve_returns * 100) if total_bh_first_serve_returns > 0 else 0

    bh_return_data.append({
        'Player': player,
        'BH 1st Serve Returns Total': total_bh_first_serve_returns,
        'BH 1st Serve Returns Win %': bh_win_percentage,
        'BH 1st Serve Returns In Play (%)': bh_in_play_percentage,
        'BH Returns Long (%)': bh_long_percentage,
        'BH Returns Net (%)': bh_net_percentage,
        'BH Returns Wide (%)': bh_wide_percentage,
        'BH Returns Deep (%)': bh_deep_percentage
    })

# Convert the list to a DataFrame
bh_return_points_df = pd.DataFrame(bh_return_data)
bh_return_points_df.head()


,Player,BH 1st Serve Returns Total,BH 1st Serve Returns Win %,BH 1st Serve Returns In Play (%),BH Returns Long (%),BH Returns Net (%),BH Returns Wide (%),BH Returns Deep (%)
0,Ben Mitchell,121,39.669421,90.909091,1.652893,2.479339,4.958678,33.057851
1,Leila Epstein,86,43.023256,69.767442,10.465116,10.465116,8.139535,40.697674
2,Lane Durkin,56,51.785714,89.285714,5.357143,3.571429,1.785714,48.214286
3,Jan Hrazdil,22,45.454545,86.363636,9.090909,4.545455,0.000000,45.454545
4,Sarah Youngberg,64,60.937500,89.062500,3.125000,4.687500,1.562500,34.375000


In [14]:
final_dataset_with_bh = pd.merge(final_data_with_placement, bh_return_points_df, on='Player', how='left')
final_dataset_with_bh.head()

,Player,Total Return Points,Return Points Won (%),Total 1st Serve Return Points,Return Points Won Against 1st Serve (%),Ad Court Total 1st Serve Returns,Ad Court 1st Serve Return Win (%),Deuce Court Total 1st Serve Returns,Deuce Court 1st Serve Return Win (%),FH 1st Serve Returns Total,...,FH Returns Net (%),FH Returns Wide (%),FH Returns Deep (%),BH 1st Serve Returns Total,BH 1st Serve Returns Win %,BH 1st Serve Returns In Play (%),BH Returns Long (%),BH Returns Net (%),BH Returns Wide (%),BH Returns Deep (%)
0,Ben Mitchell,313,46.006390,201,35.820896,108,34.259259,93,37.634409,75,...,6.250000,2.678571,33.928571,121,39.669421,90.909091,1.652893,2.479339,4.958678,33.057851
1,Leila Epstein,374,51.069519,232,46.120690,113,48.672566,119,43.697479,142,...,6.882591,2.834008,42.510121,86,43.023256,69.767442,10.465116,10.465116,8.139535,40.697674
2,Lane Durkin,195,55.897436,125,48.000000,63,44.444444,62,51.612903,66,...,10.280374,3.738318,36.448598,56,51.785714,89.285714,5.357143,3.571429,1.785714,48.214286
3,Jan Hrazdil,102,50.000000,57,43.859649,25,44.000000,32,43.750000,34,...,10.344828,6.896552,32.758621,22,45.454545,86.363636,9.090909,4.545455,0.000000,45.454545
4,Sarah Youngberg,194,59.278351,106,55.660377,60,60.000000,46,50.000000,40,...,6.818182,2.272727,43.181818,64,60.937500,89.062500,3.125000,4.687500,1.562500,34.375000


In [15]:
second_serve_data = []

for player in players:
    player_returns = data[data['B1: Returner'] == player]
    # Include both '2nd Serve Made' and '2nd Serve Ace' in the filter
    second_serve_returns = player_returns[player_returns['A2: Serve Data'].isin(['2nd Serve Made', '2nd Serve Ace'])]

    fh_second_serve_returns = second_serve_returns[second_serve_returns['B2: Return Data'].str.contains('Forehand', na=False)]
    bh_second_serve_returns = second_serve_returns[second_serve_returns['B2: Return Data'].str.contains('Backhand', na=False)]

    # Points won calculations
    fh_won_points = fh_second_serve_returns[fh_second_serve_returns['F1: Point Won'] == player]
    bh_won_points = bh_second_serve_returns[bh_second_serve_returns['F1: Point Won'] == player]

    # Compute win percentages and other statistics
    second_serve_data.append({
        'Player': player,
        '2nd Serve Returns Total': len(second_serve_returns),
        '2nd Serve Returns Win %': (len(second_serve_returns[second_serve_returns['F1: Point Won'] == player]) / len(second_serve_returns) * 100) if len(second_serve_returns) > 0 else 0,
        'FH 2nd Serve Returns Total': len(fh_second_serve_returns),
        'FH 2nd Serve Returns Win %': (len(fh_won_points) / len(fh_second_serve_returns) * 100) if len(fh_second_serve_returns) > 0 else 0,
        'BH 2nd Serve Returns Total': len(bh_second_serve_returns),
        'BH 2nd Serve Returns Win %': (len(bh_won_points) / len(bh_second_serve_returns) * 100) if len(bh_second_serve_returns) > 0 else 0
        # Continue with other statistics as needed
    })

# Convert to DataFrame
second_serve_df = pd.DataFrame(second_serve_data)
second_serve_df

,Player,2nd Serve Returns Total,2nd Serve Returns Win %,FH 2nd Serve Returns Total,FH 2nd Serve Returns Win %,BH 2nd Serve Returns Total,BH 2nd Serve Returns Win %
0,Ben Mitchell,93,56.989247,37,56.756757,56,57.142857
1,Leila Epstein,117,50.427350,105,52.380952,12,33.333333
2,Lane Durkin,55,61.818182,41,58.536585,14,71.428571
3,Jan Hrazdil,41,53.658537,24,54.166667,17,52.941176
4,Sarah Youngberg,73,56.164384,48,56.250000,25,56.000000
5,Chase Kasday,25,28.000000,20,35.000000,5,0.000000
6,Alexander Bernard,58,67.241379,39,66.666667,19,68.421053
7,Patrick Kypson,31,58.064516,13,61.538462,18,55.555556
8,Carlos Alcaraz Garfia,50,30.000000,29,31.034483,21,28.571429
9,Jackie Soloveychik,12,75.000000,0,0.000000,12,75.000000


In [16]:
# Initialize a list to hold the 2nd serve return data
second_serve_data = []

for player in players:
    player_returns = data[data['B1: Returner'] == player]
    # Consider only 2nd serves for the returns
    second_serve_returns = player_returns[player_returns['A2: Serve Data'].isin(['2nd Serve Made', '2nd Serve Ace'])]

    # Calculate total 2nd serve returns and win percentages
    total_second_serve_returns = len(second_serve_returns)
    second_serve_wins = second_serve_returns[second_serve_returns['F1: Point Won'] == player]
    second_serve_win_percentage = (len(second_serve_wins) / total_second_serve_returns * 100) if total_second_serve_returns > 0 else 0

    # Filter for forehand and backhand 2nd serve returns
    fh_second_serve_returns = second_serve_returns[second_serve_returns['B2: Return Data'].str.contains('Forehand', na=False)]
    bh_second_serve_returns = second_serve_returns[second_serve_returns['B2: Return Data'].str.contains('Backhand', na=False)]

    # Forehand statistics
    total_fh_second_serve_returns = len(fh_second_serve_returns)
    fh_second_serve_wins = fh_second_serve_returns[fh_second_serve_returns['F1: Point Won'] == player]
    fh_second_serve_win_percentage = (len(fh_second_serve_wins) / total_fh_second_serve_returns * 100) if total_fh_second_serve_returns > 0 else 0

    # Backhand statistics
    total_bh_second_serve_returns = len(bh_second_serve_returns)
    bh_second_serve_wins = bh_second_serve_returns[bh_second_serve_returns['F1: Point Won'] == player]
    bh_second_serve_win_percentage = (len(bh_second_serve_wins) / total_bh_second_serve_returns * 100) if total_bh_second_serve_returns > 0 else 0

    # Calculate placements for FH and BH returns
    fh_second_serve_in_play = fh_second_serve_returns[~fh_second_serve_returns['B2: Return Data'].str.contains('Error', na=False)]
    bh_second_serve_in_play = bh_second_serve_returns[~bh_second_serve_returns['B2: Return Data'].str.contains('Error', na=False)]

    fh_second_serve_in_play_percentage = (len(fh_second_serve_in_play) / total_fh_second_serve_returns * 100) if total_fh_second_serve_returns > 0 else 0
    bh_second_serve_in_play_percentage = (len(bh_second_serve_in_play) / total_bh_second_serve_returns * 100) if total_bh_second_serve_returns > 0 else 0

    # Add data to the list
    second_serve_data.append({
        'Player': player,
        '2nd Serve Returns Total': total_second_serve_returns,
        '2nd Serve Returns Win %': second_serve_win_percentage,
        'FH 2nd Serve Returns Total': total_fh_second_serve_returns,
        'FH 2nd Serve Returns Win %': fh_second_serve_win_percentage,
        'FH 2nd Serve Returns In Play %': fh_second_serve_in_play_percentage,
        'BH 2nd Serve Returns Total': total_bh_second_serve_returns,
        'BH 2nd Serve Returns Win %': bh_second_serve_win_percentage,
        'BH 2nd Serve Returns In Play %': bh_second_serve_in_play_percentage
        # You can add more detailed statistics such as 'Long', 'Net', 'Wide', 'Deep' etc.
    })

# Convert the list to a DataFrame
second_serve_df = pd.DataFrame(second_serve_data)
second_serve_df.head()

,Player,2nd Serve Returns Total,2nd Serve Returns Win %,FH 2nd Serve Returns Total,FH 2nd Serve Returns Win %,FH 2nd Serve Returns In Play %,BH 2nd Serve Returns Total,BH 2nd Serve Returns Win %,BH 2nd Serve Returns In Play %
0,Ben Mitchell,93,56.989247,37,56.756757,94.594595,56,57.142857,87.500000
1,Leila Epstein,117,50.427350,105,52.380952,83.809524,12,33.333333,83.333333
2,Lane Durkin,55,61.818182,41,58.536585,87.804878,14,71.428571,78.571429
3,Jan Hrazdil,41,53.658537,24,54.166667,79.166667,17,52.941176,82.352941
4,Sarah Youngberg,73,56.164384,48,56.250000,87.500000,25,56.000000,92.000000


In [17]:
second_serve_data = []

for player in players:
    player_returns = data[data['B1: Returner'] == player]
    second_serve_returns = player_returns[player_returns['A2: Serve Data'].isin(['2nd Serve Made', '2nd Serve Ace'])]

    # General 2nd serve returns statistics
    total_second_serve_returns = len(second_serve_returns)
    second_serve_wins = second_serve_returns[second_serve_returns['F1: Point Won'] == player]
    second_serve_win_percentage = (len(second_serve_wins) / total_second_serve_returns * 100) if total_second_serve_returns > 0 else 0

    # Forehand 2nd serve returns
    fh_second_serve_returns = second_serve_returns[second_serve_returns['B2: Return Data'].str.contains('Forehand', na=False)]
    total_fh_second_serve_returns = len(fh_second_serve_returns)
    fh_second_serve_wins = fh_second_serve_returns[fh_second_serve_returns['F1: Point Won'] == player]
    fh_second_serve_win_percentage = (len(fh_second_serve_wins) / total_fh_second_serve_returns * 100) if total_fh_second_serve_returns > 0 else 0

    # Backhand 2nd serve returns
    bh_second_serve_returns = second_serve_returns[second_serve_returns['B2: Return Data'].str.contains('Backhand', na=False)]
    total_bh_second_serve_returns = len(bh_second_serve_returns)
    bh_second_serve_wins = bh_second_serve_returns[bh_second_serve_returns['F1: Point Won'] == player]
    bh_second_serve_win_percentage = (len(bh_second_serve_wins) / total_bh_second_serve_returns * 100) if total_bh_second_serve_returns > 0 else 0

    # Detailed placement statistics for FH and BH
    fh_long = fh_second_serve_returns['B3: Return Placement'].str.contains('Long', na=False).sum()
    fh_net = fh_second_serve_returns['B3: Return Placement'].str.contains('Net', na=False).sum()
    fh_wide = fh_second_serve_returns['B3: Return Placement'].str.contains('Wide', na=False).sum()
    fh_deep = fh_second_serve_returns['B3: Return Placement'].str.contains('Deep', na=False).sum()
    bh_long = bh_second_serve_returns['B3: Return Placement'].str.contains('Long', na=False).sum()
    bh_net = bh_second_serve_returns['B3: Return Placement'].str.contains('Net', na=False).sum()
    bh_wide = bh_second_serve_returns['B3: Return Placement'].str.contains('Wide', na=False).sum()
    bh_deep = bh_second_serve_returns['B3: Return Placement'].str.contains('Deep', na=False).sum()

    # Add data to the list
    second_serve_data.append({
        'Player': player,
        '2nd Serve Returns Total': total_second_serve_returns,
        '2nd Serve Returns Win %': second_serve_win_percentage,
        'FH 2nd Serve Returns Total': total_fh_second_serve_returns,
        'FH 2nd Serve Returns Win %': fh_second_serve_win_percentage,
        'FH Returns Long % (2nd Serve)': (fh_long / total_fh_second_serve_returns * 100) if total_fh_second_serve_returns > 0 else 0,
        'FH Returns Net % (2nd Serve)': (fh_net / total_fh_second_serve_returns * 100) if total_fh_second_serve_returns > 0 else 0,
        'FH Returns Wide % (2nd Serve)': (fh_wide / total_fh_second_serve_returns * 100) if total_fh_second_serve_returns > 0 else 0,
        'FH Returns Deep % (2nd Serve)': (fh_deep / total_fh_second_serve_returns * 100) if total_fh_second_serve_returns > 0 else 0,
        'BH 2nd Serve Returns Total': total_bh_second_serve_returns,
        'BH 2nd Serve Returns Win %': bh_second_serve_win_percentage,
        'BH Returns Long % (2nd Serve)': (bh_long / total_bh_second_serve_returns * 100) if total_bh_second_serve_returns > 0 else 0,
        'BH Returns Net % (2nd Serve)': (bh_net / total_bh_second_serve_returns * 100) if total_bh_second_serve_returns > 0 else 0,
        'BH Returns Wide % (2nd Serve)': (bh_wide / total_bh_second_serve_returns * 100) if total_bh_second_serve_returns > 0 else 0,
        'BH Returns Deep % (2nd Serve)': (bh_deep / total_bh_second_serve_returns * 100) if total_bh_second_serve_returns > 0 else 0
    })

# Convert the list to a DataFrame
second_serve_df = pd.DataFrame(second_serve_data)
second_serve_df

,Player,2nd Serve Returns Total,2nd Serve Returns Win %,FH 2nd Serve Returns Total,FH 2nd Serve Returns Win %,FH Returns Long % (2nd Serve),FH Returns Net % (2nd Serve),FH Returns Wide % (2nd Serve),FH Returns Deep % (2nd Serve),BH 2nd Serve Returns Total,BH 2nd Serve Returns Win %,BH Returns Long % (2nd Serve),BH Returns Net % (2nd Serve),BH Returns Wide % (2nd Serve),BH Returns Deep % (2nd Serve)
0,Ben Mitchell,93,56.989247,37,56.756757,2.702703,2.702703,0.000000,51.351351,56,57.142857,8.928571,0.000000,3.571429,57.142857
1,Leila Epstein,117,50.427350,105,52.380952,5.714286,7.619048,2.857143,45.714286,12,33.333333,8.333333,8.333333,0.000000,25.000000
2,Lane Durkin,55,61.818182,41,58.536585,2.439024,2.439024,7.317073,41.463415,14,71.428571,7.142857,7.142857,7.142857,28.571429
3,Jan Hrazdil,41,53.658537,24,54.166667,8.333333,8.333333,4.166667,41.666667,17,52.941176,11.764706,0.000000,5.882353,58.823529
4,Sarah Youngberg,73,56.164384,48,56.250000,6.250000,2.083333,4.166667,54.166667,25,56.000000,4.000000,4.000000,0.000000,28.000000
5,Chase Kasday,25,28.000000,20,35.000000,5.000000,10.000000,10.000000,40.000000,5,0.000000,0.000000,0.000000,0.000000,60.000000
6,Alexander Bernard,58,67.241379,39,66.666667,2.564103,0.000000,2.564103,46.153846,19,68.421053,0.000000,0.000000,0.000000,57.894737
7,Patrick Kypson,31,58.064516,13,61.538462,23.076923,0.000000,0.000000,30.769231,18,55.555556,5.555556,0.000000,5.555556,44.444444
8,Carlos Alcaraz Garfia,50,30.000000,29,31.034483,20.689655,6.896552,6.896552,24.137931,21,28.571429,4.761905,0.000000,4.761905,38.095238
9,Jackie Soloveychik,12,75.000000,0,0.000000,0.000000,0.000000,0.000000,0.000000,12,75.000000,8.333333,0.000000,0.000000,33.333333


In [18]:
# Assuming 'final_data_with_placement' is your existing dataframe with 1st serve statistics
# and 'second_serve_df' is your newly created dataframe with 2nd serve statistics
final_dataset = pd.merge(final_data_with_placement, second_serve_df, on='Player', how='left')
final_dataset

,Player,Total Return Points,Return Points Won (%),Total 1st Serve Return Points,Return Points Won Against 1st Serve (%),Ad Court Total 1st Serve Returns,Ad Court 1st Serve Return Win (%),Deuce Court Total 1st Serve Returns,Deuce Court 1st Serve Return Win (%),FH 1st Serve Returns Total,...,FH Returns Long % (2nd Serve),FH Returns Net % (2nd Serve),FH Returns Wide % (2nd Serve),FH Returns Deep % (2nd Serve),BH 2nd Serve Returns Total,BH 2nd Serve Returns Win %,BH Returns Long % (2nd Serve),BH Returns Net % (2nd Serve),BH Returns Wide % (2nd Serve),BH Returns Deep % (2nd Serve)
0,Ben Mitchell,313,46.006390,201,35.820896,108,34.259259,93,37.634409,75,...,2.702703,2.702703,0.000000,51.351351,56,57.142857,8.928571,0.000000,3.571429,57.142857
1,Leila Epstein,374,51.069519,232,46.120690,113,48.672566,119,43.697479,142,...,5.714286,7.619048,2.857143,45.714286,12,33.333333,8.333333,8.333333,0.000000,25.000000
2,Lane Durkin,195,55.897436,125,48.000000,63,44.444444,62,51.612903,66,...,2.439024,2.439024,7.317073,41.463415,14,71.428571,7.142857,7.142857,7.142857,28.571429
3,Jan Hrazdil,102,50.000000,57,43.859649,25,44.000000,32,43.750000,34,...,8.333333,8.333333,4.166667,41.666667,17,52.941176,11.764706,0.000000,5.882353,58.823529
4,Sarah Youngberg,194,59.278351,106,55.660377,60,60.000000,46,50.000000,40,...,6.250000,2.083333,4.166667,54.166667,25,56.000000,4.000000,4.000000,0.000000,28.000000
5,Chase Kasday,123,41.463415,97,44.329897,50,54.000000,47,34.042553,47,...,5.000000,10.000000,10.000000,40.000000,5,0.000000,0.000000,0.000000,0.000000,60.000000
6,Alexander Bernard,169,50.887574,106,39.622642,46,32.608696,60,45.000000,47,...,2.564103,0.000000,2.564103,46.153846,19,68.421053,0.000000,0.000000,0.000000,57.894737
7,Patrick Kypson,70,38.571429,37,18.918919,21,23.809524,16,12.500000,17,...,23.076923,0.000000,0.000000,30.769231,18,55.555556,5.555556,0.000000,5.555556,44.444444
8,Carlos Alcaraz Garfia,128,34.375000,73,32.876712,38,36.842105,35,28.571429,41,...,20.689655,6.896552,6.896552,24.137931,21,28.571429,4.761905,0.000000,4.761905,38.095238
9,Jackie Soloveychik,51,64.705882,36,58.333333,17,58.823529,19,57.894737,10,...,0.000000,0.000000,0.000000,0.000000,12,75.000000,8.333333,0.000000,0.000000,33.333333


In [19]:
break_point_data = []

for player in players:
    player_returns = data[data['B1: Returner'] == player]
    # Filter for break point chances
    break_points = player_returns[player_returns['F4: Break Point'] == 'Break Point']

    # Count total break point chances and conversions
    total_break_points = len(break_points)
    converted_break_points = len(break_points[break_points['F1: Point Won'] == player])

    # Calculate break point conversion percentage
    conversion_percentage = (converted_break_points / total_break_points * 100) if total_break_points > 0 else 0

    break_point_data.append({
        'Player': player,
        'Total Break Point Chances': total_break_points,
        'Break Points Converted': converted_break_points,
        'Break Point Conversion %': conversion_percentage
    })

# Convert the list to a DataFrame
break_point_df = pd.DataFrame(break_point_data)
break_point_df

,Player,Total Break Point Chances,Break Points Converted,Break Point Conversion %
0,Ben Mitchell,51,22,43.137255
1,Leila Epstein,57,32,56.140351
2,Lane Durkin,36,23,63.888889
3,Jan Hrazdil,13,7,53.846154
4,Sarah Youngberg,37,24,64.864865
5,Chase Kasday,15,8,53.333333
6,Alexander Bernard,29,14,48.275862
7,Patrick Kypson,7,2,28.571429
8,Carlos Alcaraz Garfia,10,4,40.000000
9,Jackie Soloveychik,9,7,77.777778


In [20]:
final_dataset = pd.merge(final_dataset, break_point_df, on='Player', how='left')
final_dataset

,Player,Total Return Points,Return Points Won (%),Total 1st Serve Return Points,Return Points Won Against 1st Serve (%),Ad Court Total 1st Serve Returns,Ad Court 1st Serve Return Win (%),Deuce Court Total 1st Serve Returns,Deuce Court 1st Serve Return Win (%),FH 1st Serve Returns Total,...,FH Returns Deep % (2nd Serve),BH 2nd Serve Returns Total,BH 2nd Serve Returns Win %,BH Returns Long % (2nd Serve),BH Returns Net % (2nd Serve),BH Returns Wide % (2nd Serve),BH Returns Deep % (2nd Serve),Total Break Point Chances,Break Points Converted,Break Point Conversion %
0,Ben Mitchell,313,46.006390,201,35.820896,108,34.259259,93,37.634409,75,...,51.351351,56,57.142857,8.928571,0.000000,3.571429,57.142857,51,22,43.137255
1,Leila Epstein,374,51.069519,232,46.120690,113,48.672566,119,43.697479,142,...,45.714286,12,33.333333,8.333333,8.333333,0.000000,25.000000,57,32,56.140351
2,Lane Durkin,195,55.897436,125,48.000000,63,44.444444,62,51.612903,66,...,41.463415,14,71.428571,7.142857,7.142857,7.142857,28.571429,36,23,63.888889
3,Jan Hrazdil,102,50.000000,57,43.859649,25,44.000000,32,43.750000,34,...,41.666667,17,52.941176,11.764706,0.000000,5.882353,58.823529,13,7,53.846154
4,Sarah Youngberg,194,59.278351,106,55.660377,60,60.000000,46,50.000000,40,...,54.166667,25,56.000000,4.000000,4.000000,0.000000,28.000000,37,24,64.864865
5,Chase Kasday,123,41.463415,97,44.329897,50,54.000000,47,34.042553,47,...,40.000000,5,0.000000,0.000000,0.000000,0.000000,60.000000,15,8,53.333333
6,Alexander Bernard,169,50.887574,106,39.622642,46,32.608696,60,45.000000,47,...,46.153846,19,68.421053,0.000000,0.000000,0.000000,57.894737,29,14,48.275862
7,Patrick Kypson,70,38.571429,37,18.918919,21,23.809524,16,12.500000,17,...,30.769231,18,55.555556,5.555556,0.000000,5.555556,44.444444,7,2,28.571429
8,Carlos Alcaraz Garfia,128,34.375000,73,32.876712,38,36.842105,35,28.571429,41,...,24.137931,21,28.571429,4.761905,0.000000,4.761905,38.095238,10,4,40.000000
9,Jackie Soloveychik,51,64.705882,36,58.333333,17,58.823529,19,57.894737,10,...,0.000000,12,75.000000,8.333333,0.000000,0.000000,33.333333,9,7,77.777778


In [21]:
final_dataset.head()

,Player,Total Return Points,Return Points Won (%),Total 1st Serve Return Points,Return Points Won Against 1st Serve (%),Ad Court Total 1st Serve Returns,Ad Court 1st Serve Return Win (%),Deuce Court Total 1st Serve Returns,Deuce Court 1st Serve Return Win (%),FH 1st Serve Returns Total,...,FH Returns Deep % (2nd Serve),BH 2nd Serve Returns Total,BH 2nd Serve Returns Win %,BH Returns Long % (2nd Serve),BH Returns Net % (2nd Serve),BH Returns Wide % (2nd Serve),BH Returns Deep % (2nd Serve),Total Break Point Chances,Break Points Converted,Break Point Conversion %
0,Ben Mitchell,313,46.006390,201,35.820896,108,34.259259,93,37.634409,75,...,51.351351,56,57.142857,8.928571,0.000000,3.571429,57.142857,51,22,43.137255
1,Leila Epstein,374,51.069519,232,46.120690,113,48.672566,119,43.697479,142,...,45.714286,12,33.333333,8.333333,8.333333,0.000000,25.000000,57,32,56.140351
2,Lane Durkin,195,55.897436,125,48.000000,63,44.444444,62,51.612903,66,...,41.463415,14,71.428571,7.142857,7.142857,7.142857,28.571429,36,23,63.888889
3,Jan Hrazdil,102,50.000000,57,43.859649,25,44.000000,32,43.750000,34,...,41.666667,17,52.941176,11.764706,0.000000,5.882353,58.823529,13,7,53.846154
4,Sarah Youngberg,194,59.278351,106,55.660377,60,60.000000,46,50.000000,40,...,54.166667,25,56.000000,4.000000,4.000000,0.000000,28.000000,37,24,64.864865


In [22]:
final_dataset.columns

Index(['Player', 'Total Return Points', 'Return Points Won (%)',
       'Total 1st Serve Return Points',
       'Return Points Won Against 1st Serve (%)',
       'Ad Court Total 1st Serve Returns', 'Ad Court 1st Serve Return Win (%)',
       'Deuce Court Total 1st Serve Returns',
       'Deuce Court 1st Serve Return Win (%)', 'FH 1st Serve Returns Total',
       'FH 1st Serve Returns Win %', 'FH 1st Serve Returns In Play (%)',
       'FH Returns Long (%)', 'FH Returns Net (%)', 'FH Returns Wide (%)',
       'FH Returns Deep (%)', '2nd Serve Returns Total',
       '2nd Serve Returns Win %', 'FH 2nd Serve Returns Total',
       'FH 2nd Serve Returns Win %', 'FH Returns Long % (2nd Serve)',
       'FH Returns Net % (2nd Serve)', 'FH Returns Wide % (2nd Serve)',
       'FH Returns Deep % (2nd Serve)', 'BH 2nd Serve Returns Total',
       'BH 2nd Serve Returns Win %', 'BH Returns Long % (2nd Serve)',
       'BH Returns Net % (2nd Serve)', 'BH Returns Wide % (2nd Serve)',
       'BH Returns 

In [23]:
first_serve_data = []

for player in players:
    player_returns = data[data['B1: Returner'] == player]
    first_serve_returns = player_returns[player_returns['A2: Serve Data'].isin(['1st Serve Made', '1st Serve Ace'])]

    # Forehand 1st serve return stats
    fh_first_serve_returns = first_serve_returns[first_serve_returns['B2: Return Data'].str.contains('Forehand', na=False)]
    bh_first_serve_returns = first_serve_returns[first_serve_returns['B2: Return Data'].str.contains('Backhand', na=False)]

    # Calculate placements for FH 1st serve returns
    fh_long = fh_first_serve_returns['B3: Return Placement'].str.contains('Long', na=False).sum()
    fh_net = fh_first_serve_returns['B3: Return Placement'].str.contains('Net', na=False).sum()
    fh_wide = fh_first_serve_returns['B3: Return Placement'].str.contains('Wide', na=False).sum()
    fh_deep = fh_first_serve_returns['B3: Return Placement'].str.contains('Deep', na=False).sum()

    # Calculate placements for BH 1st serve returns
    bh_long = bh_first_serve_returns['B3: Return Placement'].str.contains('Long', na=False).sum()
    bh_net = bh_first_serve_returns['B3: Return Placement'].str.contains('Net', na=False).sum()
    bh_wide = bh_first_serve_returns['B3: Return Placement'].str.contains('Wide', na=False).sum()
    bh_deep = bh_first_serve_returns['B3: Return Placement'].str.contains('Deep', na=False).sum()

    total_fh_first_serve_returns = len(fh_first_serve_returns)
    total_bh_first_serve_returns = len(bh_first_serve_returns)

    first_serve_data.append({
        'Player': player,
        'FH Returns Long % (1st Serve)': (fh_long / total_fh_first_serve_returns * 100) if total_fh_first_serve_returns > 0 else 0,
        'FH Returns Net % (1st Serve)': (fh_net / total_fh_first_serve_returns * 100) if total_fh_first_serve_returns > 0 else 0,
        'FH Returns Wide % (1st Serve)': (fh_wide / total_fh_first_serve_returns * 100) if total_fh_first_serve_returns > 0 else 0,
        'FH Returns Deep % (1st Serve)': (fh_deep / total_fh_first_serve_returns * 100) if total_fh_first_serve_returns > 0 else 0,
        'BH 1st Serve Returns Total': total_bh_first_serve_returns,
        'BH 1st Serve Returns Win %': (len(bh_first_serve_returns[bh_first_serve_returns['F1: Point Won'] == player]) / total_bh_first_serve_returns * 100) if total_bh_first_serve_returns > 0 else 0,
        'BH Returns Long % (1st Serve)': (bh_long / total_bh_first_serve_returns * 100) if total_bh_first_serve_returns > 0 else 0,
        'BH Returns Net % (1st Serve)': (bh_net / total_bh_first_serve_returns * 100) if total_bh_first_serve_returns > 0 else 0,
        'BH Returns Wide % (1st Serve)': (bh_wide / total_bh_first_serve_returns * 100) if total_bh_first_serve_returns > 0 else 0,
        'BH Returns Deep % (1st Serve)': (bh_deep / total_bh_first_serve_returns * 100) if total_bh_first_serve_returns > 0 else 0
    })

# Convert the list to a DataFrame
first_serve_df = pd.DataFrame(first_serve_data)


In [24]:
# Assuming 'final_data_with_2nd_serve' is your existing comprehensive DataFrame
final_dataset = pd.merge(final_dataset, first_serve_df, on='Player', how='left')


In [25]:
final_dataset

,Player,Total Return Points,Return Points Won (%),Total 1st Serve Return Points,Return Points Won Against 1st Serve (%),Ad Court Total 1st Serve Returns,Ad Court 1st Serve Return Win (%),Deuce Court Total 1st Serve Returns,Deuce Court 1st Serve Return Win (%),FH 1st Serve Returns Total,...,FH Returns Long % (1st Serve),FH Returns Net % (1st Serve),FH Returns Wide % (1st Serve),FH Returns Deep % (1st Serve),BH 1st Serve Returns Total,BH 1st Serve Returns Win %,BH Returns Long % (1st Serve),BH Returns Net % (1st Serve),BH Returns Wide % (1st Serve),BH Returns Deep % (1st Serve)
0,Ben Mitchell,313,46.006390,201,35.820896,108,34.259259,93,37.634409,75,...,8.000000,8.000000,4.000000,25.333333,121,39.669421,1.652893,2.479339,4.958678,33.057851
1,Leila Epstein,374,51.069519,232,46.120690,113,48.672566,119,43.697479,142,...,5.633803,6.338028,2.816901,40.140845,86,43.023256,10.465116,10.465116,8.139535,40.697674
2,Lane Durkin,195,55.897436,125,48.000000,63,44.444444,62,51.612903,66,...,7.575758,15.151515,1.515152,33.333333,56,51.785714,5.357143,3.571429,1.785714,48.214286
3,Jan Hrazdil,102,50.000000,57,43.859649,25,44.000000,32,43.750000,34,...,11.764706,11.764706,8.823529,26.470588,22,45.454545,9.090909,4.545455,0.000000,45.454545
4,Sarah Youngberg,194,59.278351,106,55.660377,60,60.000000,46,50.000000,40,...,0.000000,12.500000,0.000000,30.000000,64,60.937500,3.125000,4.687500,1.562500,34.375000
5,Chase Kasday,123,41.463415,97,44.329897,50,54.000000,47,34.042553,47,...,8.510638,6.382979,0.000000,46.808511,50,48.000000,8.000000,10.000000,2.000000,40.000000
6,Alexander Bernard,169,50.887574,106,39.622642,46,32.608696,60,45.000000,47,...,8.510638,2.127660,2.127660,40.425532,58,39.655172,0.000000,8.620690,3.448276,36.206897
7,Patrick Kypson,70,38.571429,37,18.918919,21,23.809524,16,12.500000,17,...,17.647059,17.647059,0.000000,35.294118,15,26.666667,13.333333,13.333333,0.000000,33.333333
8,Carlos Alcaraz Garfia,128,34.375000,73,32.876712,38,36.842105,35,28.571429,41,...,17.073171,14.634146,4.878049,31.707317,29,41.379310,20.689655,10.344828,6.896552,20.689655
9,Jackie Soloveychik,51,64.705882,36,58.333333,17,58.823529,19,57.894737,10,...,0.000000,0.000000,0.000000,60.000000,26,57.692308,3.846154,7.692308,3.846154,61.538462


In [26]:
# Initialize a list to hold the double faults against data for each player
double_faults_against_data = []

for player in players:
    player_returns = data[data['B1: Returner'] == player]
    # Filter for points where there was a double fault
    double_faults_against = player_returns[player_returns['A2: Serve Data'].str.contains('Double Fault', na=False)]

    # Count the double faults against the player
    total_double_faults_against = len(double_faults_against)

    double_faults_against_data.append({
        'Player': player,
        'Double Faults Against': total_double_faults_against
    })

# Convert the list to a DataFrame
double_faults_against_df = pd.DataFrame(double_faults_against_data)
double_faults_against_df

,Player,Double Faults Against
0,Ben Mitchell,19
1,Leila Epstein,25
2,Lane Durkin,15
3,Jan Hrazdil,4
4,Sarah Youngberg,15
5,Chase Kasday,1
6,Alexander Bernard,5
7,Patrick Kypson,2
8,Carlos Alcaraz Garfia,5
9,Jackie Soloveychik,3


In [27]:
def rearrange_columns(final_dataset, new_order):
    return final_dataset[new_order]


In [28]:
final_dataset.columns

Index(['Player', 'Total Return Points', 'Return Points Won (%)',
       'Total 1st Serve Return Points',
       'Return Points Won Against 1st Serve (%)',
       'Ad Court Total 1st Serve Returns', 'Ad Court 1st Serve Return Win (%)',
       'Deuce Court Total 1st Serve Returns',
       'Deuce Court 1st Serve Return Win (%)', 'FH 1st Serve Returns Total',
       'FH 1st Serve Returns Win %', 'FH 1st Serve Returns In Play (%)',
       'FH Returns Long (%)', 'FH Returns Net (%)', 'FH Returns Wide (%)',
       'FH Returns Deep (%)', '2nd Serve Returns Total',
       '2nd Serve Returns Win %', 'FH 2nd Serve Returns Total',
       'FH 2nd Serve Returns Win %', 'FH Returns Long % (2nd Serve)',
       'FH Returns Net % (2nd Serve)', 'FH Returns Wide % (2nd Serve)',
       'FH Returns Deep % (2nd Serve)', 'BH 2nd Serve Returns Total',
       'BH 2nd Serve Returns Win %', 'BH Returns Long % (2nd Serve)',
       'BH Returns Net % (2nd Serve)', 'BH Returns Wide % (2nd Serve)',
       'BH Returns 

In [29]:
# Define the function to rearrange columns
def rearrange_columns(final_dataset, new_order):
    return final_dataset[new_order]

# Define the new order of columns
new_order = ['Player', 'Total Return Points', 'Return Points Won (%)',
             'Total 1st Serve Return Points', 'Return Points Won Against 1st Serve (%)',
             'Deuce Court Total 1st Serve Returns', 'Deuce Court 1st Serve Return Win (%)',
             'Ad Court Total 1st Serve Returns', 'Ad Court 1st Serve Return Win (%)',
             'FH 1st Serve Returns Total', 'FH 1st Serve Returns Win %', 'FH 1st Serve Returns In Play (%)', 'FH Returns Long (%)', 'FH Returns Net (%)', 'FH Returns Wide (%)','FH Returns Deep (%)',
             'FH Returns Long % (1st Serve)', 'FH Returns Net % (1st Serve)',
             'FH Returns Wide % (1st Serve)', 'FH Returns Deep % (1st Serve)',
             'BH 1st Serve Returns Total', 'BH 1st Serve Returns Win %',
             'BH Returns Long % (1st Serve)', 'BH Returns Net % (1st Serve)',
             'BH Returns Wide % (1st Serve)', 'BH Returns Deep % (1st Serve)',
             '2nd Serve Returns Total', '2nd Serve Returns Win %',
             'FH 2nd Serve Returns Total', 'FH 2nd Serve Returns Win %', 'FH Returns Long % (2nd Serve)',
             'FH Returns Net % (2nd Serve)', 'FH Returns Wide % (2nd Serve)', 'FH Returns Deep % (2nd Serve)',
             'BH 2nd Serve Returns Total', 'BH 2nd Serve Returns Win %', 'BH Returns Long % (2nd Serve)',
             'BH Returns Net % (2nd Serve)', 'BH Returns Wide % (2nd Serve)', 'BH Returns Deep % (2nd Serve)',
             'Total Break Point Chances', 'Break Points Converted', 'Break Point Conversion %']

# Assuming 'final_dataset' is your DataFrame loaded with data
final_dataset = rearrange_columns(final_dataset, new_order)


In [30]:
final_dataset = final_dataset.round(2)

In [31]:
final_dataset

,Player,Total Return Points,Return Points Won (%),Total 1st Serve Return Points,Return Points Won Against 1st Serve (%),Deuce Court Total 1st Serve Returns,Deuce Court 1st Serve Return Win (%),Ad Court Total 1st Serve Returns,Ad Court 1st Serve Return Win (%),FH 1st Serve Returns Total,...,FH Returns Deep % (2nd Serve),BH 2nd Serve Returns Total,BH 2nd Serve Returns Win %,BH Returns Long % (2nd Serve),BH Returns Net % (2nd Serve),BH Returns Wide % (2nd Serve),BH Returns Deep % (2nd Serve),Total Break Point Chances,Break Points Converted,Break Point Conversion %
0,Ben Mitchell,313,46.01,201,35.82,93,37.63,108,34.26,75,...,51.35,56,57.14,8.93,0.00,3.57,57.14,51,22,43.14
1,Leila Epstein,374,51.07,232,46.12,119,43.70,113,48.67,142,...,45.71,12,33.33,8.33,8.33,0.00,25.00,57,32,56.14
2,Lane Durkin,195,55.90,125,48.00,62,51.61,63,44.44,66,...,41.46,14,71.43,7.14,7.14,7.14,28.57,36,23,63.89
3,Jan Hrazdil,102,50.00,57,43.86,32,43.75,25,44.00,34,...,41.67,17,52.94,11.76,0.00,5.88,58.82,13,7,53.85
4,Sarah Youngberg,194,59.28,106,55.66,46,50.00,60,60.00,40,...,54.17,25,56.00,4.00,4.00,0.00,28.00,37,24,64.86
5,Chase Kasday,123,41.46,97,44.33,47,34.04,50,54.00,47,...,40.00,5,0.00,0.00,0.00,0.00,60.00,15,8,53.33
6,Alexander Bernard,169,50.89,106,39.62,60,45.00,46,32.61,47,...,46.15,19,68.42,0.00,0.00,0.00,57.89,29,14,48.28
7,Patrick Kypson,70,38.57,37,18.92,16,12.50,21,23.81,17,...,30.77,18,55.56,5.56,0.00,5.56,44.44,7,2,28.57
8,Carlos Alcaraz Garfia,128,34.38,73,32.88,35,28.57,38,36.84,41,...,24.14,21,28.57,4.76,0.00,4.76,38.10,10,4,40.00
9,Jackie Soloveychik,51,64.71,36,58.33,19,57.89,17,58.82,10,...,0.00,12,75.00,8.33,0.00,0.00,33.33,9,7,77.78


In [32]:
# save the DataFrame to a CSV file (for verification or local use)
final_dataset.to_csv('/Users/lkimball/Desktop/Wesleyan_Tennis_Analytics/returner_df.csv', index=True)